<a href="https://colab.research.google.com/github/kroschenko/multi_domain_model/blob/main/MultiDomainModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 0: SETUP & IMPORTS
!pip install -q transformers accelerate evaluate torchmetrics wandb librosa opencv-python seaborn scikit-learn
import os, sys, json, hashlib, tempfile, warnings, gc, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Tuple, Union
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, AutoProcessor, Wav2Vec2Model
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.stats import wilcoxon
from sklearn.model_selection import GroupKFold
from scipy import stats
import matplotlib.pyplot as plt, seaborn as sns, librosa, cv2
from tqdm import tqdm
warnings.filterwarnings('ignore')

# === AUTO-BACKUP SETUP ===
from google.colab import drive
import shutil, time, os
from pathlib import Path

# Подключить Drive
drive.mount('/content/drive', force_remount=False)

# Папка для бэкапов
BACKUP_DIR = Path("/content/drive/MyDrive/erc_project")
BACKUP_DIR.mkdir(exist_ok=True)

def save_to_drive(filename: str):
    """Сохранить файл на Google Drive"""
    src = Path(f"./{filename}")
    if src.exists():
        shutil.copy2(src, BACKUP_DIR / filename)
        print(f"Saved: {filename} → {BACKUP_DIR / filename}")

def backup_cache():
    """Сохранить кэш фич на Drive"""
    cache_src = Path("./feat_cache")
    cache_dst = BACKUP_DIR / "feat_cache"
    if cache_src.exists():
        shutil.copytree(cache_src, cache_dst, dirs_exist_ok=True)
        n_files = len(list(cache_src.rglob("*.npy")))
        size_gb = sum(f.stat().st_size for f in cache_src.rglob("*.npy")) / 1e9
        print(f"💾 Cached {n_files} files ({size_gb:.2f} GB) to Drive")

# Восстановить кэш если есть
if (BACKUP_DIR / "feat_cache").exists():
    print("🔄 Restoring cache from Drive...")
    shutil.copytree(BACKUP_DIR / "feat_cache", "./feat_cache", dirs_exist_ok=True)
    n_files = len(list(Path("./feat_cache").rglob("*.npy")))
    print(f"✅ Restored {n_files} cached features")
else:
    print("⚠️ No cached features on Drive — will extract from scratch")

print("✅ Backup system ready!")

def move_batch_to_device(batch: Dict, device: torch.device) -> Dict:
    """Recursively move all tensors in a batch dict to the specified device"""
    return {
        k: v.to(device) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

# Reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Force float32 for stability with Deberta + AMP
torch.set_default_dtype(torch.float32)
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"🔬 Device: {device} | PyTorch: {torch.__version__}")

In [ ]:
# CELL 1: CONFIGURATION

@dataclass
class PhDConfig:
    # Data
    max_train_samples: Optional[int] = None
    max_test_samples: Optional[int] = None
    max_frames: int = 8
    audio_duration: float = 4.0
    max_utterances: int = 5  # Context window size

    # Model
    text_model: str = "microsoft/deberta-v3-base"
    freeze_text: bool = True
    hidden_dim: int = 256
    dropout: float = 0.2
    num_speakers: int = 20

    # Modalities
    use_audio: bool = True
    use_video: bool = True
    use_temporal: bool = True
    use_dialogue_temporal: bool = True

    # Fusion
    fusion_type: str = "adaptive_gate"  # adaptive_gate | concat | cross_attn_only

    # Training
    batch_size: int = 6
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 0.01
    early_stopping_patience: int = 5
    missing_modality_p: float = 0.15
    uncertainty_loss_weight: float = 0.5

    # Experiment
    n_folds: int = 3
    n_seeds: int = 3
    experiment_name: str = "meld_adaptive_uncertainty_v3"
    cache_dir: str = "./feat_cache"

config = PhDConfig()
print("Configuration loaded")

In [ ]:
# CELL 2: FEATURE EXTRACTORS WITH CACHING (wav2vec2 + ResNet)

class AudioExtractor:
    def __init__(self, model_name="facebook/wav2vec2-base-960h"):
        print(f"🎵 Loading Wav2Vec2: {model_name}")
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = Wav2Vec2Model.from_pretrained(model_name)
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad = False
        self.model.to(device)
        self.audio_dim = 768  # Оставляем 768, проекция будет в модели

    def extract(self, audio_path: str, use_cache=True, cache_key=None, cache_dir="./feat_cache") -> np.ndarray:
        if use_cache and cache_key:
            cache_file = Path(cache_dir) / f"audio_{cache_key}.npy"
            if cache_file.exists(): return np.load(cache_file)
        try:
            speech, sr = librosa.load(audio_path, sr=16000, duration=config.audio_duration)
            inputs = self.processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                embeddings = self.model(**inputs).last_hidden_state  # [B, T, 768]
                # ✅ Улучшенный pooling: [CLS] + mean + std
                cls_token = embeddings[:, 0]  # [B, 768]
                mean_pool = embeddings.mean(dim=1)  # [B, 768]
                std_pool = embeddings.std(dim=1)  # [B, 768]
                features = torch.cat([cls_token, mean_pool, std_pool], dim=-1)  # [B, 2304]
                features = features.squeeze().cpu().numpy()
            if use_cache and cache_key:
                Path(cache_dir).mkdir(exist_ok=True)
                np.save(Path(cache_dir) / f"audio_{cache_key}.npy", features)
            return features.astype(np.float32)
        except Exception as e:
            # Fallback: MFCC
            y, sr = librosa.load(audio_path, duration=config.audio_duration, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            return np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)]).astype(np.float32)

class VideoExtractor:
    def __init__(self):
        print("Loading ResNet-18")
        from torchvision import models, transforms
        self.resnet = models.resnet18(weights='IMAGENET1K_V1')
        self.resnet.fc = nn.Identity()
        self.resnet.eval()
        for p in self.resnet.parameters(): p.requires_grad = False
        self.resnet.to(device)
        self.transform = transforms.Compose([
            transforms.ToPILImage(), transforms.Resize((224, 224)),
            transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.video_dim = 512

    def extract(self, video_path: str, num_frames=8, use_cache=True, cache_key=None, cache_dir="./feat_cache") -> Optional[np.ndarray]:
        if use_cache and cache_key:
            cache_file = Path(cache_dir) / f"video_{cache_key}.npy"
            if cache_file.exists(): return np.load(cache_file)
        try:
            cap = cv2.VideoCapture(video_path)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total < 3: cap.release(); return None
            idxs = np.linspace(0, total-1, min(num_frames, total), dtype=int)
            feats = []
            for i in idxs:
                cap.set(cv2.CAP_PROP_POS_FRAMES, i)
                ret, frame = cap.read()
                if ret:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    img = self.transform(frame).unsqueeze(0).to(device)
                    with torch.no_grad():
                        feat = self.resnet(img).squeeze().cpu().numpy()
                    feats.append(feat)
            cap.release()
            result = np.mean(feats, axis=0) if feats else None

            if use_cache and cache_key and result is not None:
                Path(cache_dir).mkdir(exist_ok=True)
                np.save(Path(cache_dir) / f"video_{cache_key}.npy", result)
            return result
        except: return None

def get_feature_hash(video_path: str, cfg: PhDConfig) -> str:
    with open(video_path, 'rb') as f: file_hash = hashlib.md5(f.read(8192)).hexdigest()
    config_hash = hashlib.md5(json.dumps({'max_frames': cfg.max_frames, 'audio_duration': cfg.audio_duration}, sort_keys=True).encode()).hexdigest()[:8]
    return f"{file_hash}_{config_hash}"

In [ ]:
# CELL 3: UNIFIED DATASET (MELD/IEMOCAP, Context-Aware, Dict Output)

class UnifiedERC_Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, label_encoder: Optional[List[str]] = None,
                 config: PhDConfig = None, cache_dir: str = "./feat_cache",
                 audio_ext: Optional[AudioExtractor] = None, video_ext: Optional[VideoExtractor] = None,
                 video_base_path: str = None):
        self.df = df.reset_index(drop=True)
        self.config = config or PhDConfig()
        self.cache_dir = cache_dir
        Path(cache_dir).mkdir(exist_ok=True)
        self.video_base_path = video_base_path
        self.audio_ext = audio_ext
        self.video_ext = video_ext

        # Classes & encoders
        self.classes = sorted(df["Emotion"].unique()) if "Emotion" in df.columns else sorted(df["label"].unique())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.speaker_map = {s:i for i,s in enumerate(sorted(df["Speaker"].unique()))} if "Speaker" in df.columns else {"U": 0}

        # Precompute dialogue context (window-based)
        self.contexts, self.context_speakers, self.current_indices = [], [], []
        window = config.max_utterances // 2 if config.max_utterances > 1 else 1

        for idx, row in self.df.iterrows():
            dia, utt = row["Dialogue_ID"], row["Utterance_ID"]
            ctx_mask = (df["Dialogue_ID"] == dia) & (df["Utterance_ID"] >= max(0, utt - window)) & (df["Utterance_ID"] <= utt + window)
            ctx_rows = df[ctx_mask].sort_values("Utterance_ID")

            texts = [f"[{r.get('Speaker','U')}] {r.get('Utterance', r.get('text',''))}" for _, r in ctx_rows.iterrows()]
            speakers = [self.speaker_map.get(r.get('Speaker','U'), 0) for _, r in ctx_rows.iterrows()]

            # ✅ FIX: Pad/truncate speakers to exactly max_utterances
            original_len = len(speakers)
            while len(speakers) < config.max_utterances:
                speakers.append(-1)  # -1 = padding token (invalid speaker ID)
            speakers = speakers[:config.max_utterances]  # Truncate if too long

            self.contexts.append(" [SEP] ".join(texts))
            self.context_speakers.append(torch.tensor(speakers, dtype=torch.long))  # Store as padded tensor
            self.current_indices.append(min(original_len // 2, config.max_utterances - 1))  # Clamp to valid range

        # FIX: Pre-extract ALL features during __init__ (not lazy loading)
        self._preextract_all_features()

    def _preextract_all_features(self):
        """Pre-extract and cache features for all samples to avoid DataLoader race conditions"""
        print(f"🔄 Pre-extracting features for {len(self.df)} samples...")

        for idx, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Feature Extraction", leave=False):
            uid = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}"

            # Get video path
            video_path = row.get("video_path")
            if video_path is None and self.video_base_path:
                video_path = f"{self.video_base_path}/dia{int(row['Dialogue_ID'])}_utt{int(row['Utterance_ID'])}.mp4"

            #  Extract audio feature
            if self.audio_ext:
                audio_file = Path(self.cache_dir) / f"{uid}_audio.npy"
                if not audio_file.exists():
                    if video_path and os.path.exists(video_path):
                        try:
                            # Extract audio from video using ffmpeg
                            with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as tmp:
                                tmp_path = tmp.name
                            cmd = [
                                'ffmpeg', '-i', video_path, '-vn', '-acodec', 'pcm_s16le',
                                '-ar', '16000', '-ac', '1', tmp_path, '-y', '-loglevel', 'error'
                            ]
                            subprocess.run(cmd, check=True, timeout=60, capture_output=True)
                            feat = self.audio_ext.extract(tmp_path, use_cache=True, cache_key=uid, cache_dir=self.cache_dir)
                            os.unlink(tmp_path)
                        except Exception as e:
                            print(f"⚠️ Audio extract failed for {uid}: {e}")
                            feat = np.zeros(768, dtype=np.float32)  # Fallback
                    else:
                        feat = np.zeros(768, dtype=np.float32)  # No video → zero audio
                    # Ensure saved
                    np.save(audio_file, feat.astype(np.float32))

            # Extract video feature
            if self.video_ext:
                video_file = Path(self.cache_dir) / f"{uid}_video.npy"
                if not video_file.exists():
                    if video_path and os.path.exists(video_path):
                        try:
                            feat = self.video_ext.extract(video_path, num_frames=self.config.max_frames,
                                                         use_cache=True, cache_key=uid, cache_dir=self.cache_dir)
                            if feat is None:
                                feat = np.zeros(512, dtype=np.float32)
                        except Exception as e:
                            print(f"Video extract failed for {uid}: {e}")
                            feat = np.zeros(512, dtype=np.float32)
                    else:
                        feat = np.zeros(512, dtype=np.float32)
                    np.save(video_file, feat.astype(np.float32))

        print("✅ Feature pre-extraction complete!")

        # ✅ АВТОСОХРАНЕНИЕ КЭША
        if Path(self.cache_dir).exists():
            n_files = len(list(Path(self.cache_dir).rglob("*.npy")))
            size_gb = sum(f.stat().st_size for f in Path(self.cache_dir).rglob("*.npy")) / 1e9
            print(f"💾 Backing up {n_files} features ({size_gb:.2f} GB) to Drive...")
            backup_cache()  # ← ВЫЗОВ ФУНКЦИИ

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        uid = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}"

        # Load cached features (with fallback)
        try:
            audio = torch.from_numpy(np.load(Path(self.cache_dir) / f"{uid}_audio.npy")).float()
        except FileNotFoundError:
            audio = torch.zeros(2304 if self.audio_ext else 768, dtype=torch.float32)  # ✅ Поддерживаем новый размер

        try:
            video = torch.from_numpy(np.load(Path(self.cache_dir) / f"{uid}_video.npy")).float()
        except FileNotFoundError:
            video = torch.zeros(512, dtype=torch.float32)

        # ✅ Нормализация модальностей (Fix 3)
        if audio.numel() > 0:
            audio = (audio - audio.mean()) / (audio.std() + 1e-8)
        if video.numel() > 0:
            video = (video - video.mean()) / (video.std() + 1e-8)

        context_speakers = self.context_speakers[idx]
        current_idx = torch.tensor(self.current_indices[idx], dtype=torch.long)

        return {
            "text": self.contexts[idx],
            "audio": audio,
            "video": video,
            "speaker": torch.tensor(self.speaker_map.get(row.get("Speaker", "U"), 0), dtype=torch.long),
            "labels": torch.tensor(self.class_to_idx[row.get("Emotion", row.get("label"))], dtype=torch.long),
            "context_speakers": context_speakers,
            "current_idx": current_idx,
            "uid": uid
        }


# ✅ COLLATE FUNCTION

def erc_collate_fn(batch: List[Dict]) -> Dict[str, Union[List, torch.Tensor]]:
    """
    Collate function для ERC датасета с разнородными данными.

    Вход: список из B сэмплов, каждый — dict с ключами:
        • text: str
        • audio: Tensor[768]
        • video: Tensor[512]
        • speaker: int
        • labels: int
        • context_speakers: Tensor[L] (переменная длина)
        • current_idx: int
        • uid: str

    Выход: батчированный dict:
        • text: List[str] (B элементов)
        • audio: Tensor[B, 768]
        • video: Tensor[B, 512]
        • speaker: Tensor[B]
        • labels: Tensor[B]
        • context_speakers: Tensor[B, L_max] (pad до макс. длины в батче)
        • current_idx: Tensor[B]
        • uid: List[str] (B элементов)
    """
    # Текстовые поля и UID: оставляем как списки строк (для tokenizer'а)
    texts = [b["text"] for b in batch]
    uids = [b["uid"] for b in batch]

    # Фиксированные тензоры: просто стекаем
    audio = torch.stack([b["audio"] for b in batch])           # [B, 768]
    video = torch.stack([b["video"] for b in batch])           # [B, 512]
    speaker = torch.stack([b["speaker"] for b in batch])       # [B]
    labels = torch.stack([b["labels"] for b in batch])         # [B]
    current_idx = torch.stack([b["current_idx"] for b in batch])  # [B]

    # Переменная длина: context_speakers → pad до max_len в батче
    ctx_speakers_list = [b["context_speakers"] for b in batch]  # List[Tensor[L_i]]
    max_len = max(t.size(0) for t in ctx_speakers_list)
    ctx_speakers_padded = []
    for t in ctx_speakers_list:
        if t.size(0) < max_len:
            # Pad справа нулями (значение 0 = "unknown speaker")
            pad = torch.zeros(max_len - t.size(0), dtype=t.dtype, device=t.device)
            t = torch.cat([t, pad], dim=0)
        ctx_speakers_padded.append(t)
    context_speakers = torch.stack(ctx_speakers_padded)  # [B, max_len]

    return {
        "text": texts,
        "audio": audio,
        "video": video,
        "speaker": speaker,
        "labels": labels,
        "context_speakers": context_speakers,
        "current_idx": current_idx,
        "uid": uids
    }

In [ ]:
# CELL 4: ADAPTIVE GATED FUSION

class AdaptiveGatedFusion(nn.Module):
    """
    Stabilized confidence-gated cross-modal attention.
    """
    def __init__(self, dim: int, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.quality_proj = nn.Linear(dim, 1)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        # ✅ Добавляем learnable bias для gate
        self.gate_bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, text_q: torch.Tensor, mod_kv: torch.Tensor, log_var: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B = text_q.size(0)

        # Quality score с стабилизацией
        quality = torch.sigmoid(self.quality_proj(mod_kv) + self.gate_bias)  # [B, 1]

        # Cross-attention
        attn_out, _ = self.attn(text_q.unsqueeze(1), mod_kv.unsqueeze(1), mod_kv.unsqueeze(1))
        attended = self.norm(text_q + self.dropout(attn_out.squeeze(1)))

        # ✅ Stabilized uncertainty weighting (Fix 3)
        uncertainty_weight = 1.0 / (1.0 + torch.exp(torch.clamp(log_var, min=-5, max=5)))
        uncertainty_weight = torch.clamp(uncertainty_weight, min=0.15, max=0.85)

        gate = quality * uncertainty_weight
        fused = gate * attended + (1 - gate) * text_q

        return fused, gate.squeeze(-1)  # [B,D], [B]

In [ ]:
# CELL 5: DIALOGUE CONTEXT ENCODER (Utterance-Level Temporal)

class DialogueContextEncoder(nn.Module):
    """
    Models sequence of utterances in dialogue with speaker-aware attention.
    Input: batch with pre-computed utterance embeddings OR text contexts.
    """
    def __init__(self, config: PhDConfig, max_utterances: int = 5):
        super().__init__()
        self.max_utterances = max_utterances
        self.hidden_dim = config.hidden_dim

        # ✅ Projection: DeBERTa output (768) → hidden_dim (256)
        self.utterance_proj = nn.Linear(768, config.hidden_dim)

        self.pos_emb = nn.Embedding(max_utterances, config.hidden_dim)
        self.role_emb = nn.Embedding(2, config.hidden_dim)  # binary: same/different speaker

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.hidden_dim, nhead=4,
            dim_feedforward=config.hidden_dim*2,
            dropout=config.dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.pool_proj = nn.Linear(config.hidden_dim, config.hidden_dim)

        self.utterance_encoder = AutoModel.from_pretrained(config.text_model)
        self.utterance_tok = AutoTokenizer.from_pretrained(config.text_model)

    def forward(self, batch: Dict, text_cls: torch.Tensor, device: torch.device) -> torch.Tensor:
        B = text_cls.size(0)
        context_texts = batch["text"]  # List[str]
        context_speakers = batch["context_speakers"]  # [B, seq_len] - VARIABLE!
        current_idx = batch["current_idx"]  # [B]

        # === Step 1: Encode individual utterances ===
        utterance_embs = []
        for ctx in context_texts:
            utts = ctx.split(" [SEP] ")
            utt_embs = []
            for utt in utts:
                tok = self.utterance_tok(utt.strip(), padding=True, truncation=True,
                                        max_length=64, return_tensors="pt")
                tok = {k: v.to(device) for k, v in tok.items()}
                with torch.no_grad():
                    emb = self.utterance_encoder(**tok).last_hidden_state[:, 0]  # [1, 768]
                    emb = self.utterance_proj(emb)  # ✅ Project to hidden_dim: [1, 256]
                utt_embs.append(emb.squeeze(0))  # [256]

            # Pad/truncate to exactly max_utterances
            while len(utt_embs) < self.max_utterances:
                utt_embs.append(torch.zeros(self.hidden_dim, device=device))
            utt_embs = utt_embs[:self.max_utterances]  # Truncate if too long
            utterance_embs.append(torch.stack(utt_embs))  # [max_utterances, hidden_dim]

        utterance_embs = torch.stack(utterance_embs)  # ✅ [B, max_utterances, hidden_dim]
        target_len = utterance_embs.size(1)  # Should be max_utterances

        # === Step 2: ✅ FIX - Align context_speakers to target_len ===
        if context_speakers.size(1) != target_len:
            if context_speakers.size(1) < target_len:
                # Pad with 0 (valid speaker ID, won't affect role_emb lookup)
                pad_len = target_len - context_speakers.size(1)
                padding = torch.zeros((B, pad_len), dtype=torch.long, device=device)
                context_speakers = torch.cat([context_speakers, padding], dim=1)
            else:
                # Truncate if too long
                context_speakers = context_speakers[:, :target_len]

        # === Step 3: Positional + role embeddings ===
        positions = torch.arange(target_len, device=device).unsqueeze(0).expand(B, -1)
        utterance_embs = utterance_embs + self.pos_emb(positions)

        # Clamp current_idx to valid range and get current speaker
        current_idx = current_idx.clamp(0, target_len - 1)
        current_speaker = context_speakers[torch.arange(B), current_idx]  # [B]

        # Role mask: 1 if same speaker as current utterance, 0 otherwise
        role_mask = (context_speakers == current_speaker.unsqueeze(1)).long()  # [B, target_len]
        utterance_embs = utterance_embs + self.role_emb(role_mask)  # ✅ Shapes now match!

        # === Step 4: Transformer with causal mask ===
        mask = torch.triu(torch.ones(target_len, target_len, device=device), diagonal=1).bool()
        out = self.transformer(utterance_embs)  # [B, target_len, hidden_dim]

        # === Step 5: Weighted pooling around current utterance ===
        weights = torch.exp(-torch.abs(
            torch.arange(target_len, device=device).unsqueeze(0) -
            current_idx.unsqueeze(1)
        ).float())  # [B, target_len]
        pooled = (out * weights.unsqueeze(-1)).sum(dim=1) / weights.sum(dim=1, keepdim=True)  # [B, hidden_dim]

        return self.pool_proj(pooled)  # [B, hidden_dim]

In [ ]:

# CELL 6: PhD HYBRID MODEL (Full Architecture)

class PhDHybridModel(nn.Module):
    def __init__(self, config: PhDConfig, num_classes: int = 7):
        super().__init__()
        self.config = config

        self.text_tok = AutoTokenizer.from_pretrained(config.text_model)
        self.text_enc = AutoModel.from_pretrained(config.text_model)

        if config.freeze_text:
            for p in self.text_enc.parameters():
                p.requires_grad = False
            # ✅ Разморозить последние 2 слоя для fine-tuning
            if not config.freeze_text:
                for layer in self.text_enc.encoder.layer[-2:]:
                    for p in layer.parameters():
                        p.requires_grad = True

        # ✅ Проекция: 2304 (audio) / 768 (text) → hidden_dim
        self.text_proj = nn.Linear(768, config.hidden_dim)
        self.audio_proj = nn.Linear(2304, config.hidden_dim) if config.use_audio else None  # ✅ Новый размер аудио

        self.speaker_emb = nn.Embedding(config.num_speakers, config.hidden_dim)

        self.video_proj = nn.Sequential(
            nn.Linear(512, config.hidden_dim), nn.ReLU(), nn.Dropout(config.dropout)
        ) if config.use_video else None

        # Fusion
        self.fusion_audio = AdaptiveGatedFusion(config.hidden_dim) if (config.use_audio and config.fusion_type == "adaptive_gate") else None
        self.fusion_video = AdaptiveGatedFusion(config.hidden_dim) if (config.use_video and config.fusion_type == "adaptive_gate") else None

        # Temporal
        self.temporal = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=config.hidden_dim, nhead=4,
                                       dim_feedforward=config.hidden_dim*2,
                                       dropout=config.dropout, batch_first=True),
            num_layers=1
        ) if config.use_temporal else None

        self.dialogue_encoder = DialogueContextEncoder(config, max_utterances=config.max_utterances) if config.use_dialogue_temporal else None

        # Uncertainty head
        self.unc_head = nn.Sequential(
            nn.Linear(config.hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        # ✅ Classifier: [text, audio, video] → 3 * hidden_dim
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_dim * 3, config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, num_classes)
        )

    def forward(self, batch: Dict, training: bool = False):
        B = batch["labels"].size(0)
        device = batch["labels"].device

        audio = batch["audio"].to(device) if self.config.use_audio else torch.zeros(B, 768, device=device, dtype=torch.float32)
        video = batch["video"].to(device) if self.config.use_video else torch.zeros(B, 512, device=device, dtype=torch.float32)

        speaker = batch["speaker"].to(device)
        # ✅ FIX: Clamp speaker IDs to valid embedding range
        speaker = torch.clamp(speaker, 0, self.config.num_speakers - 1)

        # Text encoding
        tok = self.text_tok(batch["text"], padding=True, truncation=True, max_length=128, return_tensors="pt")
        tok = {k: v.to(device) for k, v in tok.items()}

        text_out = self.text_enc(**tok)
        text_cls = text_out.last_hidden_state[:, 0]           # [B, 768]
        text_cls = self.text_proj(text_cls)                   # [B, 256]
        text_cls = text_cls + self.speaker_emb(speaker)       # [B, 256]

        # Project modalities
        a_feat = self.audio_proj(audio) if self.audio_proj is not None else torch.zeros_like(text_cls)
        v_feat = self.video_proj(video) if self.video_proj is not None else torch.zeros_like(text_cls)

        # Within-utterance temporal
        if self.temporal is not None:
            seq = torch.stack([text_cls, a_feat, v_feat], dim=1)   # [B, 3, 256]
            seq = self.temporal(seq)
            text_cls = seq.mean(dim=1)

        # Dialogue-level temporal (отключено в quick_test)
        if self.dialogue_encoder is not None and self.config.use_dialogue_temporal:
            text_cls = self.dialogue_encoder(batch, text_cls, device)

        # Uncertainty estimation
        log_var = torch.clamp(self.unc_head(text_cls), min=-5.0, max=2.0)   # [B, 1]

        # Fusion
        gates = {}
        fused_text = text_cls.clone()

        if self.config.fusion_type == "adaptive_gate":
            if self.fusion_audio is not None and self.config.use_audio:
                fused_text, gate_a = self.fusion_audio(fused_text, a_feat, log_var)
                gates["audio"] = gate_a
            if self.fusion_video is not None and self.config.use_video:
                fused_text, gate_v = self.fusion_video(fused_text, v_feat, log_var)
                gates["video"] = gate_v

        # Final classifier input - concatenation of all three modalities
        fused = torch.cat([text_cls, a_feat, v_feat], dim=1)   # [B, 768]

        logits = self.classifier(fused)

        return logits, log_var, gates

In [ ]:
# CELL 7: TRAINING LOOP (AMP, Uncertainty Loss, Early Stopping)

def train_phd(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
              config: PhDConfig, seed: int, train_ds) -> Tuple[nn.Module, float]:

    model.to(device)
    model = model.float()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

    # ✅ Fix 2: Warmup scheduler
    from transformers import get_cosine_schedule_with_warmup
    total_steps = config.epochs * len(train_loader)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps
    )

    scaler = GradScaler()
    best_f1, best_state, patience = 0.0, None, 0

    # ✅ Fix 1: Правильные веса классов через sklearn
    from sklearn.utils.class_weight import compute_class_weight
    if hasattr(train_ds, 'df') and 'Emotion' in train_ds.df.columns:
        emotion_col = 'Emotion' if 'Emotion' in train_ds.df.columns else 'label'
        train_labels = train_ds.df[emotion_col].map(train_ds.class_to_idx).values
    else:
        train_labels = []
        for batch in train_loader:
            train_labels.extend(batch["labels"].cpu().numpy())
        train_labels = np.array(train_labels)

    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
    print(f"📊 Class weights: {class_weights}")

    # ✅ Fix 4: Проверка forward pass перед тренировкой
    model.eval()
    with torch.no_grad():
        dummy_batch = next(iter(train_loader))
        dummy_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in dummy_batch.items()}
        try:
            logits, log_var, gates = model(dummy_batch, training=False)
            print(f"✓ Forward OK: logits.shape={logits.shape}, nan={torch.isnan(logits).any().item()}")
            print(f"  Logits range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")
        except Exception as e:
            print(f"✗ Forward failed: {e}")
            raise
    model.train()

    for epoch in range(config.epochs):
        model.train(); train_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Seed {seed} | Epoch {epoch+1}", leave=False):
            batch = {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }

            optimizer.zero_grad()
            with autocast():
                logits, log_var, _ = model(batch, training=True)
                ce_loss = F.cross_entropy(logits, batch["labels"], weight=class_weights_tensor)

                if config.uncertainty_loss_weight > 0:
                    prec = torch.exp(-torch.clamp(log_var.squeeze(-1), min=-5, max=5))
                    unc_loss = prec * ce_loss + log_var.squeeze(-1)
                    loss = ce_loss + config.uncertainty_loss_weight * unc_loss.mean()
                else:
                    loss = ce_loss

            scaler.scale(loss).backward()

            # ✅ Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer); scaler.update()
            scheduler.step()  # ✅ Шаг планировщика внутри цикла
            train_loss += loss.item()

        # ========== VALIDATION ==========
        model.eval(); val_preds, val_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {
                    k: v.to(device) if isinstance(v, torch.Tensor) else v
                    for k, v in batch.items()
                }
                logits, _, _ = model(batch, training=False)
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_true.extend(batch["labels"].cpu().numpy())

        val_f1 = f1_score(val_true, val_preds, average="macro", zero_division=0)

        # ✅ Debug print для первого epoch
        if epoch == 0:
            print(f"\n🔍 Debug epoch 0:")
            print(f"   Train loss: {train_loss/len(train_loader):.4f}")
            print(f"   Val F1: {val_f1:.4f}")
            print(f"   Pred distribution: {np.bincount(val_preds, minlength=len(train_ds.classes))}")
            print(f"   True distribution: {np.bincount(val_true, minlength=len(train_ds.classes))}")

        # ========== CHECKPOINT SAVING ==========
        if epoch % 2 == 0 or epoch == config.epochs - 1:
            ckpt_path = f'{config.cache_dir}/checkpoint_seed{seed}_epoch{epoch}.pt'
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': asdict(config),
                'seed': seed,
            }, ckpt_path)
            if Path(ckpt_path).exists() and 'BACKUP_DIR' in globals():
                shutil.copy2(ckpt_path, BACKUP_DIR / f'checkpoint_seed{seed}_epoch{epoch}.pt')

        # ========== EARLY STOPPING ==========
        if val_f1 > best_f1:
            best_f1, patience = val_f1, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= config.early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    if best_state:
        model.load_state_dict(best_state)

    torch.cuda.empty_cache(); gc.collect()
    return model, best_f1

In [ ]:
# CELL 8: ABLATION SUITE (Fixed, Pairwise Wilcoxon)

def run_ablation_suite(model_cls, base_config: PhDConfig, train_df: pd.DataFrame, val_df: pd.DataFrame,
                       test_df: pd.DataFrame, cache_dir: str = "./feat_cache",
                       audio_ext: Optional[AudioExtractor] = None, video_ext: Optional[VideoExtractor] = None,
                       video_base_path: str = None) -> pd.DataFrame:
    configs = {
        "Full_Adaptive": {"use_audio": True, "use_video": True, "use_temporal": True, "fusion_type": "adaptive_gate", "uncertainty_loss_weight": 0.5},
        "w/o_Temporal": {"use_audio": True, "use_video": True, "use_temporal": False, "fusion_type": "adaptive_gate", "uncertainty_loss_weight": 0.5},
        "w/o_Uncertainty": {"use_audio": True, "use_video": True, "use_temporal": True, "uncertainty_loss_weight": 0.0},
        "w/o_MissingDropout": {"use_audio": True, "use_video": True, "missing_modality_p": 0.0},
        "w/o_SpeakerEmb": {"num_speakers": 1},
        "Simple_Concat": {"fusion_type": "concat", "uncertainty_loss_weight": 0.0},
        "TextOnly_Deberta": {"use_audio": False, "use_video": False, "use_temporal": False},
    }

    all_results, baseline_preds, baseline_true = {}, None, None

    for name, cfg_overrides in configs.items():
        cfg = PhDConfig(**{**asdict(base_config), **cfg_overrides})
        print(f"\nRunning: {name}")
        seed_metrics, all_preds = [], []

        for seed in range(42, 42 + cfg.n_seeds):
            set_seed(seed)
            train_ds = UnifiedERC_Dataset(train_df, config=cfg, cache_dir=cache_dir, audio_ext=audio_ext, video_ext=video_ext, video_base_path=video_base_path)
            val_ds = UnifiedERC_Dataset(val_df, config=cfg, cache_dir=cache_dir)
            test_ds = UnifiedERC_Dataset(test_df, config=cfg, cache_dir=cache_dir)

            dl_tr = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=erc_collate_fn, num_workers=0)
            dl_vl = DataLoader(val_ds, batch_size=cfg.batch_size, collate_fn=erc_collate_fn, num_workers=0)
            dl_ts = DataLoader(test_ds, batch_size=cfg.batch_size, collate_fn=erc_collate_fn, num_workers=0)

            model = model_cls(cfg, num_classes=len(train_ds.classes))
            _, val_f1 = train_phd(model, dl_tr, dl_vl, cfg, seed, train_ds)

            y_t, y_p = [], []
            model.eval()
            with torch.no_grad():
                for b in dl_ts:
                    b = move_batch_to_device(b, device)
                    b = {
                        k: v.to(device) if isinstance(v, torch.Tensor) else v
                        for k, v in b.items()
                    }
                    logits, _, _ = model(b, training=False)
                    y_t.extend(b["labels"].cpu().numpy())
                    y_p.extend(logits.argmax(1).cpu().numpy())

            seed_metrics.append({"seed": seed, "macro_f1": f1_score(y_t, y_p, average="macro"), "weighted_f1": f1_score(y_t, y_p, average="weighted")})
            all_preds.append(np.array(y_p))
            if baseline_true is None: baseline_true = np.array(y_t)
            torch.cuda.empty_cache(); gc.collect()

        macro_scores = [m["macro_f1"] for m in seed_metrics]
        weighted_scores = [m["weighted_f1"] for m in seed_metrics]
        p_values = {}
        if name != "Full_Adaptive" and "Full_Adaptive" in all_results:
                full_preds_list = all_results["Full_Adaptive"]["preds"]  # List[np.array], one per seed
                for i, preds in enumerate(all_preds):
                    if i < len(full_preds_list) and len(preds) == len(full_preds_list[i]):
                        correct_diff = (preds == baseline_true).astype(float) - (full_preds_list[i] == baseline_true).astype(float)
                        try:
                            _, p = wilcoxon(correct_diff, alternative="greater", method="approx")
                            p_values[f"seed{i+42}"] = float(p)
                        except Exception as e:
                            p_values[f"seed{i+42}"] = "N/A"
                    else:
                        p_values[f"seed{i+42}"] = "N/A"

        all_results[name] = {
            "macro_f1": f"{np.mean(macro_scores):.3f} ± {np.std(macro_scores):.3f}",
            "weighted_f1": f"{np.mean(weighted_scores):.3f} ± {np.std(weighted_scores):.3f}",
            "p_values": p_values, "preds": all_preds
        }
        if name == "Full_Adaptive": baseline_preds = all_preds
        print(f"  ✓ Macro: {all_results[name]['macro_f1']} | Weighted: {all_results[name]['weighted_f1']}")

    return pd.DataFrame([{"Model": k, **v} for k, v in all_results.items()])

In [ ]:
# CELL 9: INTERPRETABILITY VISUALIZATIONS

def visualize_gate_analysis(model: nn.Module, test_loader: DataLoader, config: PhDConfig, emotion_labels: List[str], save_dir: str = "./viz"):
    Path(save_dir).mkdir(exist_ok=True); model.eval()
    gate_a_all, gate_v_all, y_true_all, y_pred_all = [], [], [], []

    with torch.no_grad():
        for batch in test_loader:
            logits, _, gates = model(batch, training=False)
            preds = logits.argmax(1).cpu().numpy()
            gate_a_all.extend(gates.get("audio", torch.zeros(len(preds))).cpu().numpy())
            gate_v_all.extend(gates.get("video", torch.zeros(len(preds))).cpu().numpy())
            y_true_all.extend(batch["labels"].cpu().numpy()); y_pred_all.extend(preds)

    gate_a_all, gate_v_all = np.array(gate_a_all), np.array(gate_v_all)
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)

    # Gate distribution per emotion
    plt.figure(figsize=(12, 5))
    for emo_idx, emo_name in enumerate(emotion_labels):
        mask = y_true_all == emo_idx
        if mask.sum() < 10: continue
        plt.subplot(1, 2, 1); sns.kdeplot(gate_a_all[mask], label=emo_name, fill=True, alpha=0.3)
        plt.subplot(1, 2, 2); sns.kdeplot(gate_v_all[mask], label=emo_name, fill=True, alpha=0.3)
    plt.subplot(1, 2, 1); plt.title("Audio Gate"); plt.xlabel("Gate Value"); plt.legend(fontsize=8, ncol=2)
    plt.subplot(1, 2, 2); plt.title("Video Gate"); plt.xlabel("Gate Value"); plt.legend(fontsize=8, ncol=2)
    plt.tight_layout(); plt.savefig(f"{save_dir}/gate_distribution.png", dpi=300); plt.close()

    # Confidence-conditioned accuracy
    thresholds, results = [0.3, 0.5, 0.7, 0.9], []
    for emo_idx, emo_name in enumerate(emotion_labels):
        mask = y_true_all == emo_idx

        if mask.sum() < 20:
            continue

        row = {"Emotion": emo_name, "N": int(mask.sum())}

        for thresh in thresholds:
            high_conf = gate_a_all[mask] > thresh
            if high_conf.sum() > 5:
                acc = accuracy_score(y_true_all[mask][high_conf], y_pred_all[mask][high_conf])
                row[f"AudioGate>{thresh}"] = f"{acc:.3f}"

        results.append(row)

    if results:
        pd.DataFrame(results).to_csv(f"{save_dir}/confidence_conditioned.csv", index=False)

    print(f"Visualizations saved to {save_dir}")

In [ ]:
# CELL 10: LATEX TABLE GENERATOR (Publication-Ready)

def generate_latex_table(ablation_df: pd.DataFrame, caption: str = "Ablation study on MELD test set.", label: str = "tab:ablation") -> str:
    latex = f"\\begin{{table}}[t]\n\\centering\n\\small\n\\caption{{{caption}}}\n\\label{{{label}}}\n\\begin{{tabular}}{{lcccc}}\n\\toprule\n\\textbf{{Model}} & \\textbf{{Weighted-F1}} & \\textbf{{Macro-F1}} & \\textbf{{Δ vs Concat}} & \\textbf{{p-value}} \\\\\n\\midrule\n"
    baseline_weighted = float(ablation_df[ablation_df["Model"]=="Simple_Concat"]["weighted_f1"].iloc[0].split()[0]) if "Simple_Concat" in ablation_df["Model"].values else 0.65
    for _, row in ablation_df.iterrows():
        weighted, macro = row["weighted_f1"], row["macro_f1"]
        delta = ""
        if row["Model"] != "Simple_Concat":
            curr = float(weighted.split()[0]); delta = f"+{curr - baseline_weighted:.2f}" if curr > baseline_weighted else f"{curr - baseline_weighted:.2f}"
        p_val = list(row.get("p_values", {}).values())[0] if row.get("p_values") else "—"
        latex += f"{row['Model']} & {weighted} & {macro} & {delta} & {p_val} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex


In [ ]:
# CELL 11: IEMOCAP ADAPTER (Unified Format)

def load_iemocap_to_unified(iemocap_dir: str, cache_dir: str = "./feat_cache") -> pd.DataFrame:
    """Converts IEMOCAP to UnifiedERC_Dataset format"""
    records = []
    emo_map = {"hap": "joy", "sad": "sadness", "ang": "anger", "neu": "neutral", "exc": "joy", "fru": "anger", "sur": "surprise", "dis": "disgust", "fea": "fear"}

    for session in ["Ses01", "Ses02", "Ses03"]:
        sess_path = os.path.join(iemocap_dir, session, "dialog", "transcription")
        if not os.path.exists(sess_path): continue
        for dia_file in os.listdir(sess_path):
            if not dia_file.endswith(".txt"): continue
            dia_id = f"{session}_{dia_file.replace('.txt', '')}"
            with open(os.path.join(sess_path, dia_file), 'r', encoding='utf-8') as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]
            for utt_idx, line in enumerate(lines):
                parts = line.split('[', 2)
                if len(parts) < 3: continue
                speaker = parts[1].split(']')[0]
                emo_meta, text = parts[2].split(']', 1)
                emotion = emo_map.get(emo_meta.strip().split()[0], "neutral")
                records.append({"Dialogue_ID": dia_id, "Utterance_ID": utt_idx, "Utterance": text.strip(), "Speaker": speaker, "Emotion": emotion, "uid": f"{dia_id}_utt{utt_idx}"})
    return pd.DataFrame(records)


In [ ]:
# Добавьте в конец notebook для быстрой проверки архитектуры:
def quick_test():
    """Проверяет архитектуру модели"""
    cfg = PhDConfig(
        batch_size=2,
        epochs=1,
        n_seeds=1,
        use_dialogue_temporal=False,   # отключаем тяжёлую часть
        hidden_dim=256,
        dropout=0.2
    )
    set_seed(42)

    # Создаём dummy batch
    batch = {
        "text": [
            "[S1] Hello how are you today [SEP] [S2] I'm fine thank you",
            "[S1] What do you think about this movie"
        ],
        "audio": torch.randn(2, 768, dtype=torch.float32),
        "video": torch.randn(2, 512, dtype=torch.float32),
        "speaker": torch.tensor([0, 1]),
        "labels": torch.tensor([0, 1]),
        "context_speakers": torch.tensor([[0, 1, 0, 0, 0], [1, 0, 0, 0, 0]]),
        "current_idx": torch.tensor([1, 0]),
        "uid": ["test_1", "test_2"]
    }

    model = PhDHybridModel(cfg, num_classes=2)

    # Важно: сначала переводим модель на устройство и в float32
    model = model.to(device)
    model = model.float()                    # ← обязательно

    # Теперь переводим batch на GPU
    batch = {
        "text": batch["text"],                                 # текст остаётся list[str]
        "audio": batch["audio"].to(device),
        "video": batch["video"].to(device),
        "speaker": batch["speaker"].to(device),
        "labels": batch["labels"].to(device),
        "context_speakers": batch["context_speakers"].to(device),
        "current_idx": batch["current_idx"].to(device),
        "uid": batch["uid"]
    }

    # Forward pass
    logits, log_var, gates = model(batch, training=False)

    print("✅ Quick test PASSED successfully!")
    print(f"   Logits shape: {logits.shape} | dtype: {logits.dtype}")
    print(f"   Device: {next(model.parameters()).device}")
    return model

In [ ]:
def diagnostic_check(model, train_loader, device):
    """Быстрая диагностика модели перед тренировкой"""
    model.eval()
    batch = next(iter(train_loader))
    batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    print("🔍 Diagnostic Report:")
    print(f"  Batch size: {batch['labels'].size(0)}")
    print(f"  Text length: {len(batch['text'][0])} chars")
    print(f"  Audio shape: {batch['audio'].shape}, norm: {batch['audio'].norm().item():.3f}")
    print(f"  Video shape: {batch['video'].shape}, norm: {batch['video'].norm().item():.3f}")

    with torch.no_grad():
        logits, log_var, gates = model(batch, training=False)

    print(f"  Logits: shape={logits.shape}, range=[{logits.min().item():.2f}, {logits.max().item():.2f}]")
    print(f"  Logits nan/inf: {torch.isnan(logits).any().item()}/{torch.isinf(logits).any().item()}")
    print(f"  Log_var: range=[{log_var.min().item():.2f}, {log_var.max().item():.2f}]")
    if gates:
        for k, v in gates.items():
            print(f"  Gate[{k}]: mean={v.mean().item():.3f}, std={v.std().item():.3f}")

    preds = logits.argmax(1)
    print(f"  Predictions: unique={torch.unique(preds).tolist()}, distribution={torch.bincount(preds, minlength=7).tolist()}")
    print(f"  Labels: distribution={torch.bincount(batch['labels'], minlength=7).tolist()}")
    print("✅ Diagnostic complete\n")

In [ ]:
def main():
    print("Starting ERC Pipeline")

    # 🧪 DEBUG FLAG
    DEBUG_MODE = False  # ← Меняйте здесь

    # === 1. ВСЕГДА создаём базовый конфиг ===
    config = PhDConfig()  # ← Базовый конфиг всегда создаётся!

    # === 2. Загружаем данные ===
    try:
        import kagglehub
        path = kagglehub.dataset_download("bhandariprakanda/meld-emotion-recognition")
        video_base = os.path.join(path, "MELD.Raw", "MELD.Raw")
        csv_path = os.path.join(path, "JSON files", "JSON files", "Updated CSV")

        train_df = pd.read_csv(os.path.join(csv_path, "train_sent_emo_cleaned.csv"))
        test_df = pd.read_csv(os.path.join(csv_path, "test_sent_emo_cleaned.csv"))

        train_df["video_path"] = train_df.apply(
            lambda r: os.path.join(video_base, "train", "train_splits", f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}.mp4"), axis=1)
        test_df["video_path"] = test_df.apply(
            lambda r: os.path.join(video_base, "test", "output_repeated_splits_test", f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}.mp4"), axis=1)

        val_df = train_df.sample(frac=0.15, random_state=42)
        train_df = train_df.drop(val_df.index).reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)

        print(f"📊 Loaded MELD: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

    except Exception as e:
        print(f"Using dummy data (error: {e})")
        train_df = pd.DataFrame({
            "Dialogue_ID": [1]*50 + [2]*50,
            "Utterance_ID": list(range(50))*2,
            "Utterance": ["Hello world"]*100,
            "Speaker": ["S1", "S2"]*50,
            "Emotion": ["neutral", "joy"]*50,
            "video_path": [None]*100
        })
        test_df = train_df.iloc[:20].copy()
        val_df = train_df.iloc[20:40].copy()

    # === 3. ОПЦИОНАЛЬНО: переопределяем конфиг для DEBUG режима ===
    if DEBUG_MODE:
        print("⚡ Running in DEBUG MODE (100 samples, 1 epoch, 1 seed)")

        # Подсэмплинг данных
        train_df = train_df.head(33).reset_index(drop=True)
        val_df = val_df.head(33).reset_index(drop=True) if len(val_df) > 0 else train_df.head(33).reset_index(drop=True)
        test_df = test_df.head(34).reset_index(drop=True)
        print(f"📊 Debug subsets: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

        # Переопределяем конфиг
        config = PhDConfig(
            max_train_samples=33, max_test_samples=34,
            max_frames=4, audio_duration=2.0, max_utterances=3,
            freeze_text=True, hidden_dim=64, dropout=0.1,
            use_audio=True, use_video=False,
            use_temporal=False, use_dialogue_temporal=False,
            fusion_type="concat",
            batch_size=2, epochs=1, lr=5e-5,
            early_stopping_patience=5, uncertainty_loss_weight=0.0,
            n_folds=1, n_seeds=1,
            experiment_name="debug_test",
            cache_dir="./debug_cache"
        )
        # Очистка старого кэша
        if Path(config.cache_dir).exists():
            import shutil; shutil.rmtree(config.cache_dir)
            print(f"🗑️ Cleared old debug cache: {config.cache_dir}")

    # === 4. Теперь config гарантированно определён! ===
    print(f"⚙️ Config: epochs={config.epochs}, batch={config.batch_size}, freeze_text={config.freeze_text}")

    # === EXTRACTORS ===
    audio_ext = AudioExtractor() if config.use_audio else None
    video_ext = VideoExtractor() if config.use_video else None

    print("\nRunning ablation study...")
    ablation_results = run_ablation_suite(
        PhDHybridModel, config, train_df, val_df, test_df,
        cache_dir=config.cache_dir,
        audio_ext=audio_ext,
        video_ext=video_ext,
        video_base_path=os.path.dirname(train_df["video_path"].iloc[0]) if "video_path" in train_df.columns else None
    )

    print("\nLaTeX Table:")
    print(generate_latex_table(ablation_results))
    ablation_results.to_csv("ablation_results.csv", index=False)  # ← Без "_debug" для полного запуска
    print("\n✅ Pipeline complete. Results saved to ablation_results.csv")
    # Save results
    ablation_results.to_csv("ablation_results.csv", index=False)

    # ✅ АВТОСОХРАНЕНИЕ РЕЗУЛЬТАТОВ
    save_to_drive("ablation_results.csv")
    backup_cache()  # На всякий случай — обновить кэш

    print("\n✅ Pipeline complete. Results saved locally and to Drive")
    return ablation_results

In [ ]:
model = quick_test()

In [ ]:
if __name__ == "__main__":
    main()